In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = pd.read_csv('/datasets/Churn.csv') 

In [2]:
DATA_PATH.info()
DATA_PATH.head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), object(3)
memory usage: 1.1+ MB


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2.0,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1.0,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8.0,159660.80,3,1,0,113931.57,1


In [3]:


import pandas as pd
import numpy as np

 
#    Si no existiera, intenta cargar desde el CSV original.
try:
    source_df = DATA_PATH  # ya lo tienes como DataFrame
except NameError:
    source_df = pd.read_csv('/datasets/Churn.csv')

# 1) Copia de trabajo en memoria
df = source_df.copy()

# 2) Eliminar columnas con fuga para el modelado (si están presentes)
leak_cols = ["RowNumber", "CustomerId", "Surname"]
cols_present = [c for c in leak_cols if c in df.columns]

# Guardarlas aparte por si las quieres auditar
id_frame = df[cols_present].copy() if cols_present else pd.DataFrame()

# DataFrame de modelado sin fugas
df_model = df.drop(columns=cols_present, errors="ignore")

# 3) Vista rápida
display(df_model.head(3))
print("Eliminadas (si existían):", cols_present)
print("Shape df_model:", df_model.shape)


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2.0,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1.0,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8.0,159660.80,3,1,0,113931.57,1


Eliminadas (si existían): ['RowNumber', 'CustomerId', 'Surname']
Shape df_model: (10000, 11)


In [4]:
try:
    source_df = DATA_PATH  # si ya hiciste DATA_PATH = pd.read_csv(...)
except NameError:
    source_df = pd.read_csv('/datasets/Churn.csv')

df = source_df.copy()

# 1) Quitar columnas con fuga para el modelado
leak_cols = ["RowNumber", "CustomerId", "Surname"]
cols_present = [c for c in leak_cols if c in df.columns]
id_frame = df[cols_present].copy() if cols_present else pd.DataFrame()
df_model = df.drop(columns=cols_present, errors="ignore")

# 2) Imputar Tenure + bandera
if "Tenure" in df_model.columns and df_model["Tenure"].isna().any():
    df_model["Tenure_isna"] = df_model["Tenure"].isna().astype(int)
    df_model["Tenure"] = df_model["Tenure"].fillna(df_model["Tenure"].median())
else:
    df_model["Tenure_isna"] = 0

# 3) Asegurar tipos (enteros donde aplique)
for col in ["HasCrCard", "IsActiveMember", "NumOfProducts", "Age", "CreditScore", "Exited"]:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors="coerce")
        if df_model[col].isna().any():
            df_model[col] = df_model[col].fillna(df_model[col].mode(dropna=True)[0])
        # Enteros para evitar floats tipo 0.0/1.0
        df_model[col] = df_model[col].astype(int)

# 4) Reglas de sanidad suaves (no eliminar salvo casos imposibles)
# Marca outliers, no los borres (útil para diagnóstico)
def flag_outside(s, low=None, high=None):
    m = pd.Series(False, index=s.index)
    if low is not None:  m |= s < low
    if high is not None: m |= s > high
    return m.astype(int)

df_model["Age_out"] = flag_outside(df_model["Age"], low=18, high=95) if "Age" in df_model else 0
df_model["Balance_out"] = flag_outside(df_model["Balance"], low=0) if "Balance" in df_model else 0
df_model["CreditScore_out"] = flag_outside(df_model["CreditScore"], low=250, high=950) if "CreditScore" in df_model else 0
df_model["NumOfProducts_out"] = flag_outside(df_model["NumOfProducts"], low=1) if "NumOfProducts" in df_model else 0

# 5) Banderas opcionales útiles
if "Balance" in df_model: df_model["ZeroBalance"] = (df_model["Balance"] == 0).astype(int)
if "NumOfProducts" in df_model: df_model["HighProducts"] = (df_model["NumOfProducts"] >= 3).astype(int)

# 6) Diagnósticos
print("Eliminadas (si existían):", cols_present)
if "CustomerId" in df.columns:
    print("Posibles duplicados por CustomerId:", df["CustomerId"].duplicated(keep=False).sum())

print("\nFaltantes (df_model):")
print(df_model.isna().sum().sort_values(ascending=False).head(10))

print("\nDistribución Exited:", df_model["Exited"].value_counts(normalize=True).round(3).to_dict())

for c in ["Geography", "Gender"]:
    if c in df_model.columns:
        print(f"\nValores únicos {c}:", df_model[c].unique())


Eliminadas (si existían): ['RowNumber', 'CustomerId', 'Surname']
Posibles duplicados por CustomerId: 0

Faltantes (df_model):
CreditScore          0
Geography            0
ZeroBalance          0
NumOfProducts_out    0
CreditScore_out      0
Balance_out          0
Age_out              0
Tenure_isna          0
Exited               0
EstimatedSalary      0
dtype: int64

Distribución Exited: {0: 0.796, 1: 0.204}

Valores únicos Geography: ['France' 'Spain' 'Germany']

Valores únicos Gender: ['Female' 'Male']


In [5]:
# Asumimos que vienes de la Parte 1 y ya tienes df_model listo
y = df_model["Exited"].astype(int)
X = df_model.drop(columns=["Exited"])

# Balance de clases
class_counts = y.value_counts()
class_ratio = y.value_counts(normalize=True).round(3)
print("Conteo por clase:\n", class_counts.to_dict())
print("Proporción por clase:\n", class_ratio.to_dict())


Conteo por clase:
 {0: 7963, 1: 2037}
Proporción por clase:
 {0: 0.796, 1: 0.204}


In [6]:
from sklearn.model_selection import train_test_split

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=12345, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=12345, stratify=y_train_full
)

print("Train:", X_train.shape, "Valid:", X_valid.shape, "Test:", X_test.shape)
print("Distribución y_train:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Distribución y_valid:", y_valid.value_counts(normalize=True).round(3).to_dict())
print("Distribución y_test :", y_test.value_counts(normalize=True).round(3).to_dict())

Train: (6000, 17) Valid: (2000, 17) Test: (2000, 17)
Distribución y_train: {0: 0.796, 1: 0.204}
Distribución y_valid: {0: 0.796, 1: 0.204}
Distribución y_test : {0: 0.796, 1: 0.204}


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, confusion_matrix, classification_report
import numpy as np

# Columnas numéricas / categóricas (respetando casing)
num_cols = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]
cat_cols = ["Geography", "Gender"]

num_prep = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# <-- FIX de compatibilidad: usar sparse_output si existe; si no, usar sparse=False
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

cat_prep = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", ohe)
])

preprocess = ColumnTransformer([
    ("num", num_prep, num_cols),
    ("cat", cat_prep, cat_cols)
], remainder="drop")

# Modelo base: SIN class_weight
baseline = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000))  # sin class_weight
])

baseline.fit(X_train, y_train)

# Probabilidades y predicción a umbral 0.5
proba_valid = baseline.predict_proba(X_valid)[:, 1]
pred_valid = (proba_valid >= 0.5).astype(int)

# Métricas en VALID
f1 = f1_score(y_valid, pred_valid)
auc = roc_auc_score(y_valid, proba_valid)
acc = accuracy_score(y_valid, pred_valid)
cm = confusion_matrix(y_valid, pred_valid)

print("Baseline (sin balanceo) — Valid:")
print(f"F1: {f1:.4f} | AUC-ROC: {auc:.4f} | Accuracy: {acc:.4f}")
print("Matriz de confusión:\n", cm)
print("\nReporte de clasificación:\n", classification_report(y_valid, pred_valid, digits=4))



Baseline (sin balanceo) — Valid:
F1: 0.1680 | AUC-ROC: 0.7742 | Accuracy: 0.7970
Matriz de confusión:
 [[1553   40]
 [ 366   41]]

Reporte de clasificación:
               precision    recall  f1-score   support

           0     0.8093    0.9749    0.8844      1593
           1     0.5062    0.1007    0.1680       407

    accuracy                         0.7970      2000
   macro avg     0.6577    0.5378    0.5262      2000
weighted avg     0.7476    0.7970    0.7386      2000



In [8]:
from sklearn.dummy import DummyClassifier

dummy = Pipeline([
    ("prep", baseline.named_steps["prep"]),
    ("clf", DummyClassifier(strategy="most_frequent"))
])
dummy.fit(X_train, y_train)

pred_valid_dummy = dummy.predict(X_valid)
# probas para AUC (si no están, usa la tasa de positivos como constante)
if hasattr(dummy.named_steps["clf"], "predict_proba"):
    proba_valid_dummy = dummy.predict_proba(X_valid)[:, 1]
else:
    proba_valid_dummy = np.full(len(y_valid), (y_train==1).mean())

print("\nDummy (mayoría) — Valid:")
print("F1:", f1_score(y_valid, pred_valid_dummy))
print("AUC-ROC:", roc_auc_score(y_valid, proba_valid_dummy))
print("Accuracy:", accuracy_score(y_valid, pred_valid_dummy))
print("Matriz de confusión:\n", confusion_matrix(y_valid, pred_valid_dummy))



Dummy (mayoría) — Valid:
F1: 0.0
AUC-ROC: 0.5
Accuracy: 0.7965
Matriz de confusión:
 [[1593    0]
 [ 407    0]]


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, roc_auc_score, precision_recall_curve, confusion_matrix, classification_report

# Columnas (casing original)
num_cols = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]
cat_cols = ["Geography", "Gender"]

def make_preprocess():
    num_prep = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    # Compatibilidad sklearn (sparse_output vs sparse)
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
    cat_prep = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", ohe)
    ])
    return ColumnTransformer([
        ("num", num_prep, num_cols),
        ("cat", cat_prep, cat_cols)
    ], remainder="drop")

def best_threshold_f1(y_true, proba):
    precisions, recalls, thresholds = precision_recall_curve(y_true, proba)
    # thresholds tiene len = len(precisions)-1; creamos arreglo alineado
    thrs = np.r_[0, thresholds, 1]
    f1s = 2 * (precisions*recalls) / (precisions + recalls + 1e-12)
    idx = int(np.nanargmax(f1s))
    return float(thrs[idx]), float(f1s[idx])

def evaluate_with_threshold(model, Xv, yv, threshold):
    proba = model.predict_proba(Xv)[:, 1]
    pred = (proba >= threshold).astype(int)
    return {
        "F1": f1_score(yv, pred),
        "AUC": roc_auc_score(yv, proba),
        "CM": confusion_matrix(yv, pred),
        "report": classification_report(yv, pred, digits=4)
    }

In [19]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

preprocess = make_preprocess()

# Pipelines
pipe_lr_cw = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

pipe_rf_cw = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(class_weight="balanced", random_state=12345, n_jobs=-1))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=12345)

# Grids compactos y efectivos
grid_lr = {
    "clf__C": [0.3, 1.0, 3.0]
}
grid_rf = {
    "clf__n_estimators": [300, 500],
    "clf__max_depth": [None, 10],
    "clf__min_samples_leaf": [1, 3]
}

gs_lr = GridSearchCV(pipe_lr_cw, grid_lr, scoring="f1", cv=cv, n_jobs=-1, verbose=0)
gs_rf = GridSearchCV(pipe_rf_cw, grid_rf, scoring="f1", cv=cv, n_jobs=-1, verbose=0)

# Entrenar en TRAIN
gs_lr.fit(X_train, y_train)
gs_rf.fit(X_train, y_train)

print("LR (cw) best params:", gs_lr.best_params_, "CV F1:", round(gs_lr.best_score_, 4))
print("RF (cw) best params:", gs_rf.best_params_, "CV F1:", round(gs_rf.best_score_, 4))

# Ajuste de umbral en VALID para el mejor de cada uno
proba_lr_valid = gs_lr.predict_proba(X_valid)[:, 1]
thr_lr, f1_lr_thr = best_threshold_f1(y_valid, proba_lr_valid)

proba_rf_valid = gs_rf.predict_proba(X_valid)[:, 1]
thr_rf, f1_rf_thr = best_threshold_f1(y_valid, proba_rf_valid)

print(f"LR(cw) mejor umbral={thr_lr:.3f}, F1@thr={f1_lr_thr:.4f}")
print(f"RF(cw) mejor umbral={thr_rf:.3f}, F1@thr={f1_rf_thr:.4f}")

res_lr = evaluate_with_threshold(gs_lr, X_valid, y_valid, thr_lr)
res_rf = evaluate_with_threshold(gs_rf, X_valid, y_valid, thr_rf)

print("\n[VALID] LR(cw) -> F1:", round(res_lr["F1"],4), "AUC:", round(res_lr["AUC"],4))
print("Matriz de confusión LR(cw):\n", res_lr["CM"])
print(res_lr["report"])

print("\n[VALID] RF(cw) -> F1:", round(res_rf["F1"],4), "AUC:", round(res_rf["AUC"],4))
print("Matriz de confusión RF(cw):\n", res_rf["CM"])
print(res_rf["report"])


LR (cw) best params: {'clf__C': 3.0} CV F1: 0.4791
RF (cw) best params: {'clf__max_depth': 10, 'clf__min_samples_leaf': 3, 'clf__n_estimators': 300} CV F1: 0.5886
LR(cw) mejor umbral=0.566, F1@thr=0.5451
RF(cw) mejor umbral=0.574, F1@thr=0.6189

[VALID] LR(cw) -> F1: 0.5445 AUC: 0.7772
Matriz de confusión LR(cw):
 [[1305  288]
 [ 147  260]]
              precision    recall  f1-score   support

           0     0.8988    0.8192    0.8571      1593
           1     0.4745    0.6388    0.5445       407

    accuracy                         0.7825      2000
   macro avg     0.6866    0.7290    0.7008      2000
weighted avg     0.8124    0.7825    0.7935      2000


[VALID] RF(cw) -> F1: 0.6181 AUC: 0.863
Matriz de confusión RF(cw):
 [[1459  134]
 [ 165  242]]
              precision    recall  f1-score   support

           0     0.8984    0.9159    0.9071      1593
           1     0.6436    0.5946    0.6181       407

    accuracy                         0.8505      2000
   macro avg   

In [15]:
from sklearn.linear_model import LogisticRegression

def train_on_resampled(X_tr, y_tr, X_val, y_val, sampler="down", ratio=1.0):
    X_rs, y_rs = (downsample_majority(X_tr, y_tr, ratio=ratio) 
                  if sampler=="down" else 
                  upsample_minority(X_tr, y_tr, ratio=ratio))
    pipe = Pipeline([
        ("prep", make_preprocess()),
        ("clf", LogisticRegression(max_iter=2000))  # sin class_weight
    ])
    pipe.fit(X_rs, y_rs)
    proba = pipe.predict_proba(X_val)[:, 1]
    thr, f1_thr = best_threshold_f1(y_val, proba)
    eval_res = evaluate_with_threshold(pipe, X_val, y_val, thr)
    return thr, f1_thr, eval_res, pipe

# Probar distintas razones
for sampler in ["down", "up"]:
    for ratio in [1.0, 0.8, 1.5]:  # balance 1:1, un poco más/menos
        thr, f1_thr, res, _ = train_on_resampled(X_train, y_train, X_valid, y_valid, sampler=sampler, ratio=ratio)
        print(f"\n[{sampler}sampling ratio={ratio}] Mejor umbral={thr:.3f} | F1@thr={f1_thr:.4f} | AUC={res['AUC']:.4f}")
        print("Matriz de confusión:\n", res["CM"])



[downsampling ratio=1.0] Mejor umbral=0.560 | F1@thr=0.5326 | AUC=0.7763
Matriz de confusión:
 [[1302  291]
 [ 154  253]]

[downsampling ratio=0.8] Mejor umbral=0.616 | F1@thr=0.5349 | AUC=0.7776
Matriz de confusión:
 [[1295  298]
 [ 150  257]]

[downsampling ratio=1.5] Mejor umbral=0.462 | F1@thr=0.5300 | AUC=0.7751
Matriz de confusión:
 [[1300  293]
 [ 155  252]]

[upsampling ratio=1.0] Mejor umbral=0.558 | F1@thr=0.5391 | AUC=0.7777
Matriz de confusión:
 [[1289  304]
 [ 145  262]]

[upsampling ratio=0.8] Mejor umbral=0.500 | F1@thr=0.5358 | AUC=0.7770
Matriz de confusión:
 [[1283  310]
 [ 145  262]]

[upsampling ratio=1.5] Mejor umbral=0.664 | F1@thr=0.5426 | AUC=0.7778
Matriz de confusión:
 [[1306  287]
 [ 149  258]]


In [16]:
# Ejemplo de consolidación simple:
candidates = []

# De class_weight:
candidates.append(("LR_cw", gs_lr, thr_lr, res_lr["F1"], res_lr["AUC"]))
candidates.append(("RF_cw", gs_rf, thr_rf, res_rf["F1"], res_rf["AUC"]))

# De sampling (si te gustó alguno):
#   guarda (nombre, modelo, threshold, F1_valid, AUC_valid) manualmente, por ejemplo:
# candidates.append(("LR_down_1.0", pipe_down, thr_down, f1_down, auc_down))

best = sorted(candidates, key=lambda x: (x[3], x[4]), reverse=True)[0]
best_name, best_model, best_thr, best_f1_valid, best_auc_valid = best
print("\nMEJOR en VALID:", best_name, "F1:", round(best_f1_valid,4), "AUC:", round(best_auc_valid,4))



MEJOR en VALID: RF_cw F1: 0.6181 AUC: 0.863


In [20]:
# ====== MASTER CELL: lectura, preparación, split, baseline y dummy con impresiones ======
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, confusion_matrix, classification_report

print(">> 1) Carga CSV")
df = pd.read_csv("/datasets/Churn.csv")
print("Shape:", df.shape)

# --- Preparación mínima (NO cambia casing) ---
print("\n>> 2) Preparación mínima")
leak_cols = ["RowNumber", "CustomerId", "Surname"]
cols_present = [c for c in leak_cols if c in df.columns]
df_model = df.drop(columns=cols_present, errors="ignore")

# Imputación de Tenure + bandera
if "Tenure" in df_model.columns and df_model["Tenure"].isna().any():
    df_model["Tenure_isna"] = df_model["Tenure"].isna().astype(int)
    df_model["Tenure"] = df_model["Tenure"].fillna(df_model["Tenure"].median())
else:
    df_model["Tenure_isna"] = 0

# Asegurar tipos enteros donde aplica
for col in ["HasCrCard", "IsActiveMember", "NumOfProducts", "Age", "CreditScore", "Exited"]:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors="coerce")
        if df_model[col].isna().any():
            df_model[col] = df_model[col].fillna(df_model[col].mode(dropna=True)[0])
        df_model[col] = df_model[col].astype(int)

print("Eliminadas (si existían):", cols_present)
print("Faltantes top:\n", df_model.isna().sum().sort_values(ascending=False).head(5))
print("Distribución Exited:", df_model["Exited"].value_counts(normalize=True).round(3).to_dict())

# --- Split 60/20/20 ---
print("\n>> 3) Split estratificado 60/20/20")
y = df_model["Exited"].astype(int)
X = df_model.drop(columns=["Exited"])
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=12345, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=12345, stratify=y_train_full
)
print("Shapes -> Train:", X_train.shape, "Valid:", X_valid.shape, "Test:", X_test.shape)
print("y_train dist:", y_train.value_counts(normalize=True).round(3).to_dict())
print("y_valid dist:", y_valid.value_counts(normalize=True).round(3).to_dict())
print("y_test  dist:", y_test.value_counts(normalize=True).round(3).to_dict())

# --- Prepro con fix de compatibilidad para OneHotEncoder ---
print("\n>> 4) Construir preprocess")
num_cols = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]
cat_cols = ["Geography", "Gender"]

num_prep = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

cat_prep = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", ohe)
])

preprocess = ColumnTransformer([
    ("num", num_prep, num_cols),
    ("cat", cat_prep, cat_cols)
], remainder="drop")

# --- Baseline: LR sin balanceo, umbral 0.5 ---
print("\n>> 5) Baseline (LR sin balanceo, thr=0.5)")
baseline = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000))
])
baseline.fit(X_train, y_train)
proba_valid = baseline.predict_proba(X_valid)[:, 1]
pred_valid = (proba_valid >= 0.5).astype(int)

f1 = f1_score(y_valid, pred_valid)
auc = roc_auc_score(y_valid, proba_valid)
acc = accuracy_score(y_valid, pred_valid)
cm  = confusion_matrix(y_valid, pred_valid)
print(f"Baseline — VALID -> F1: {f1:.4f} | AUC: {auc:.4f} | ACC: {acc:.4f}")
print("Matriz de confusión (VALID):\n", cm)
print("Reporte (VALID):\n", classification_report(y_valid, pred_valid, digits=4))

# --- Dummy: mayoría, usando el MISMO preprocess ---
print("\n>> 6) Dummy (mayoría)")
dummy = Pipeline([
    ("prep", baseline.named_steps["prep"]),
    ("clf", DummyClassifier(strategy="most_frequent"))
])
dummy.fit(X_train, y_train)
pred_valid_dummy = dummy.predict(X_valid)
if hasattr(dummy.named_steps["clf"], "predict_proba"):
    proba_valid_dummy = dummy.predict_proba(X_valid)[:, 1]
else:
    proba_valid_dummy = np.full(len(y_valid), (y_train==1).mean(), dtype=float)

print(f"Dummy — VALID -> F1: {f1_score(y_valid, pred_valid_dummy):.4f} | AUC: {roc_auc_score(y_valid, proba_valid_dummy):.4f} | ACC: {accuracy_score(y_valid, pred_valid_dummy):.4f}")
print("Matriz de confusión (VALID):\n", confusion_matrix(y_valid, pred_valid_dummy))


>> 1) Carga CSV
Shape: (10000, 14)

>> 2) Preparación mínima
Eliminadas (si existían): ['RowNumber', 'CustomerId', 'Surname']
Faltantes top:
 CreditScore    0
Geography      0
Gender         0
Age            0
Tenure         0
dtype: int64
Distribución Exited: {0: 0.796, 1: 0.204}

>> 3) Split estratificado 60/20/20
Shapes -> Train: (6000, 11) Valid: (2000, 11) Test: (2000, 11)
y_train dist: {0: 0.796, 1: 0.204}
y_valid dist: {0: 0.796, 1: 0.204}
y_test  dist: {0: 0.796, 1: 0.204}

>> 4) Construir preprocess

>> 5) Baseline (LR sin balanceo, thr=0.5)
Baseline — VALID -> F1: 0.1680 | AUC: 0.7742 | ACC: 0.7970
Matriz de confusión (VALID):
 [[1553   40]
 [ 366   41]]
Reporte (VALID):
               precision    recall  f1-score   support

           0     0.8093    0.9749    0.8844      1593
           1     0.5062    0.1007    0.1680       407

    accuracy                         0.7970      2000
   macro avg     0.6577    0.5378    0.5262      2000
weighted avg     0.7476    0.7970    

In [25]:
# ====== MEJORA: dos enfoques de desbalance + comparación en VALID ======
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, roc_auc_score, precision_recall_curve, confusion_matrix, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import shuffle as sk_shuffle

# --- 1) Utilidades ---
num_cols = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]
cat_cols = ["Geography", "Gender"]

def make_preprocess():
    num_prep = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    # Compatibilidad sklearn: sparse_output (nuevo) vs sparse (viejo)
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
    cat_prep = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", ohe)
    ])
    return ColumnTransformer([
        ("num", num_prep, num_cols),
        ("cat", cat_prep, cat_cols)
    ], remainder="drop")

def best_threshold_f1(y_true, proba):
    p, r, thr = precision_recall_curve(y_true, proba)
    thr_full = np.r_[0, thr, 1]
    f1s = 2*(p*r)/(p+r+1e-12)
    i = int(np.nanargmax(f1s))
    return float(thr_full[i]), float(f1s[i])

def evaluate(model, Xv, yv, thr):
    proba = model.predict_proba(Xv)[:, 1]
    pred = (proba >= thr).astype(int)
    return {
        "F1": f1_score(yv, pred),
        "AUC": roc_auc_score(yv, proba),
        "thr": thr,
        "CM": confusion_matrix(yv, pred),
        "report": classification_report(yv, pred, digits=4)
    }

# --- 2) Enfoque A: class_weight='balanced' (LR y RF) con pequeña búsqueda de hiperparámetros ---
results = []  # cada item: (name, params_dict, eval_dict)

preprocess = make_preprocess()

# Logistic Regression (balanced)
for C in [0.3, 1.0, 3.0]:
    pipe = Pipeline([
        ("prep", preprocess),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", C=C))
    ])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_valid)[:, 1]
    thr, _ = best_threshold_f1(y_valid, proba)
    ev = evaluate(pipe, X_valid, y_valid, thr)
    results.append(("LR_cw", {"C": C}, ev))

# Random Forest (balanced)
for n in [300, 500]:
    for d in [None, 10]:
        pipe = Pipeline([
            ("prep", preprocess),
            ("clf", RandomForestClassifier(
                n_estimators=n, max_depth=d, class_weight="balanced",
                random_state=12345, n_jobs=-1
            ))
        ])
        pipe.fit(X_train, y_train)
        proba = pipe.predict_proba(X_valid)[:, 1]
        thr, _ = best_threshold_f1(y_valid, proba)
        ev = evaluate(pipe, X_valid, y_valid, thr)
        results.append(("RF_cw", {"n_estimators": n, "max_depth": d}, ev))

# --- 3) Enfoque B: re-muestreo (down / up) + LR sin class_weight ---
def downsample_majority(X, y, ratio=1.0, random_state=12345):
    Xy = X.copy(); Xy["__y__"] = y.values
    maj = Xy[Xy["__y__"] == 0]; mino = Xy[Xy["__y__"] == 1]
    n_target = int(len(mino) * ratio)
    maj_ds = maj.sample(n=min(len(maj), n_target), random_state=random_state)
    Xy_ds = pd.concat([mino, maj_ds], axis=0)
    Xy_ds = sk_shuffle(Xy_ds, random_state=random_state)
    y_new = Xy_ds["__y__"].astype(int); X_new = Xy_ds.drop(columns="__y__")
    return X_new, y_new

def upsample_minority(X, y, ratio=1.0, random_state=12345):
    Xy = X.copy(); Xy["__y__"] = y.values
    maj = Xy[Xy["__y__"] == 0]; mino = Xy[Xy["__y__"] == 1]
    need = int(len(maj) * ratio) - len(mino)
    if need > 0:
        mino_extra = mino.sample(n=need, replace=True, random_state=random_state)
        Xy_us = pd.concat([maj, mino, mino_extra], axis=0)
    else:
        Xy_us = pd.concat([maj, mino], axis=0)
    Xy_us = sk_shuffle(Xy_us, random_state=random_state)
    y_new = Xy_us["__y__"].astype(int); X_new = Xy_us.drop(columns="__y__")
    return X_new, y_new

for sampler_name, sampler_fn in [("down", downsample_majority), ("up", upsample_minority)]:
    for ratio in [1.0, 0.8, 1.5]:
        X_rs, y_rs = sampler_fn(X_train, y_train, ratio=ratio)
        pipe = Pipeline([
            ("prep", make_preprocess()),
            ("clf", LogisticRegression(max_iter=2000))  # sin class_weight
        ])
        pipe.fit(X_rs, y_rs)
        proba = pipe.predict_proba(X_valid)[:, 1]
        thr, _ = best_threshold_f1(y_valid, proba)
        ev = evaluate(pipe, X_valid, y_valid, thr)
        results.append((f"LR_{sampler_name}", {"ratio": ratio}, ev))

# --- 4) RANKING ROBUSTO (con id) + selección del TOP-1 ---
rows = []
for idx, (name, params, ev) in enumerate(results):
    row = {"id": idx, "model": name, "F1_valid": float(ev["F1"]), "AUC_valid": float(ev["AUC"]), "thr_opt": float(ev["thr"])}
    for k, v in (params or {}).items():
        row[k] = v
    rows.append(row)

rank = pd.DataFrame(rows).sort_values(["F1_valid", "AUC_valid"], ascending=False).reset_index(drop=True)

print("\n=== RANKING VALID (Top 10) ===")
try:
    display(rank.head(10))
except NameError:
    print(rank.head(10).to_string(index=False))

best_id = int(rank.loc[0, "id"])
best_name, best_params, best_ev = results[best_id]

print(f"\n=== MEJOR MODELO EN VALID: {best_name} {best_params} ===")
print(f"F1_valid={best_ev['F1']:.4f} | AUC_valid={best_ev['AUC']:.4f} | thr_opt={best_ev['thr']:.4f}")
print("Matriz de confusión (VALID):\n", best_ev["CM"])
print("Reporte (VALID):\n", best_ev["report"])



=== RANKING VALID (Top 10) ===


,id,model,F1_valid,AUC_valid,thr_opt,C,n_estimators,max_depth,ratio
0,6,RF_cw,0.623295,0.860966,0.443390,NaN,500.0,10.0,NaN
1,4,RF_cw,0.620690,0.861436,0.440548,NaN,300.0,10.0,NaN
2,3,RF_cw,0.613932,0.846278,0.320000,NaN,300.0,NaN,NaN
3,5,RF_cw,0.611995,0.846765,0.334000,NaN,500.0,NaN,NaN
4,0,LR_cw,0.544503,0.777283,0.565218,0.3,NaN,NaN,NaN
5,1,LR_cw,0.544503,0.777236,0.565908,1.0,NaN,NaN,NaN
6,2,LR_cw,0.544503,0.777232,0.566110,3.0,NaN,NaN,NaN
7,12,LR_up,0.542017,0.777843,0.663764,NaN,NaN,NaN,1.5
8,10,LR_up,0.538541,0.777739,0.558413,NaN,NaN,NaN,1.0
9,11,LR_up,0.535240,0.776996,0.499697,NaN,NaN,NaN,0.8



=== MEJOR MODELO EN VALID: RF_cw {'n_estimators': 500, 'max_depth': 10} ===
F1_valid=0.6233 | AUC_valid=0.8610 | thr_opt=0.4434
Matriz de confusión (VALID):
 [[1344  249]
 [ 110  297]]
Reporte (VALID):
               precision    recall  f1-score   support

           0     0.9243    0.8437    0.8822      1593
           1     0.5440    0.7297    0.6233       407

    accuracy                         0.8205      2000
   macro avg     0.7342    0.7867    0.7527      2000
weighted avg     0.8469    0.8205    0.8295      2000



In [26]:
# ====== FINAL: Reentrenar mejor modelo en TRAIN+VALID y evaluar en TEST ======
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, confusion_matrix, classification_report
from sklearn.utils.validation import check_is_fitted

# 1) Unir train+valid
X_trfull = pd.concat([X_train, X_valid], axis=0).reset_index(drop=True)
y_trfull = pd.concat([y_train, y_valid], axis=0).reset_index(drop=True)

# 2) Preprocesamiento (compatible con cualquier versión de sklearn)
num_cols = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]
cat_cols = ["Geography", "Gender"]

num_prep = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
cat_prep = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", ohe)
])
preprocess = ColumnTransformer([
    ("num", num_prep, num_cols),
    ("cat", cat_prep, cat_cols)
], remainder="drop")

# 3) Mejor modelo encontrado en VALID
best_model = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=500, max_depth=10,
        class_weight="balanced", random_state=12345, n_jobs=-1
    ))
])

# 4) Entrenar en TRAIN+VALID
best_model.fit(X_trfull, y_trfull)

# 5) Elegir umbral que maximiza F1 en TRAIN+VALID
from sklearn.metrics import precision_recall_curve
proba_trfull = best_model.predict_proba(X_trfull)[:, 1]
p, r, thr = precision_recall_curve(y_trfull, proba_trfull)
thr_full = np.r_[0, thr, 1]
f1s = 2*(p*r)/(p+r+1e-12)
thr_opt = float(thr_full[int(np.nanargmax(f1s))])

print(f">> Umbral óptimo (train+valid): {thr_opt:.4f}")

# 6) Evaluar en TEST
proba_test = best_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= thr_opt).astype(int)

f1 = f1_score(y_test, pred_test)
auc = roc_auc_score(y_test, proba_test)
acc = accuracy_score(y_test, pred_test)
cm  = confusion_matrix(y_test, pred_test)

print("\n=== RESULTADOS EN TEST ===")
print(f"F1: {f1:.4f} | AUC-ROC: {auc:.4f} | Accuracy: {acc:.4f}")
print("Matriz de confusión:\n", cm)
print("\nReporte de clasificación:\n", classification_report(y_test, pred_test, digits=4))

# 7) Recordatorio de objetivo
print("\n>> Meta F1 ≥ 0.59 —", "ALCANZADA ✅" if f1 >= 0.59 else "AÚN NO ❌")


>> Umbral óptimo (train+valid): 0.5931

=== RESULTADOS EN TEST ===
F1: 0.6158 | AUC-ROC: 0.8621 | Accuracy: 0.8540
Matriz de confusión:
 [[1474  119]
 [ 173  234]]

Reporte de clasificación:
               precision    recall  f1-score   support

           0     0.8950    0.9253    0.9099      1593
           1     0.6629    0.5749    0.6158       407

    accuracy                         0.8540      2000
   macro avg     0.7789    0.7501    0.7628      2000
weighted avg     0.8477    0.8540    0.8500      2000


>> Meta F1 ≥ 0.59 — ALCANZADA ✅
